<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signals

- `age_days`: number of days from the last update to the decision date.
- `decline_ratio`: `(H1 − H2) / H1`, the ratio of the impressions drop between the two halves of month 3.


- `age_norm` = `min(age_days / 365, 1)`
- `decline_norm` = `min(max(decline_ratio, 0), 1)`

### Score

`score = w1 × age_norm + w2 × decline_norm`

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"

START, END  = "2026-03-01", "2026-03-31"
MIN_IMPR_M3 = 100   # الحد الأدنى للـ impressions في شهر 3

con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '{START}' AND DATE '{END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR_M3}
""")

slice_df = con.execute("SELECT * FROM content_slice").df()
print(len(slice_df))


HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101168


In [2]:
slice_df

,client_hash_id,content_hash_id
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,client_73cda7b4e4f265ea,content_905aa32a0230694e
2,client_73cda7b4e4f265ea,content_05434271b257bb68
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c
4,client_73cda7b4e4f265ea,content_2662845f598544ef
...,...,...
101163,client_3ffa76342f366962,content_48c9b57be92bb11b
101164,client_3ffa76342f366962,content_52c98b3f26951f3b
101165,client_1a730cb2640a1abf,content_051c8125a3d20e83
101166,client_1a730cb2640a1abf,content_45155b648fff917d


In [3]:
stale_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    s.client_hash_id AS client,
    DATE_DIFF(
        'day',
        CAST(
            CASE
                WHEN d.content_updated_date IS NOT NULL
                     AND d.content_updated_date <= DATE '2026-03-31'
                THEN d.content_updated_date
                ELSE d.content_created_date
            END AS DATE
        ),
        DATE '2026-03-31'
    ) AS stale_days
FROM read_parquet('{SRC_CONTENT}') d
JOIN slice_df s USING (content_hash_id)
WHERE CASE
    WHEN d.content_updated_date IS NOT NULL
         AND d.content_updated_date <= DATE '2026-03-31'
    THEN d.content_updated_date
    ELSE d.content_created_date
END IS NOT NULL
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
stale_df

,content,client,stale_days
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,396
1,content_905aa32a0230694e,client_73cda7b4e4f265ea,396
2,content_05434271b257bb68,client_73cda7b4e4f265ea,396
3,content_d056587ff7faca0c,client_73cda7b4e4f265ea,396
4,content_2662845f598544ef,client_73cda7b4e4f265ea,396
...,...,...,...
101163,content_48c9b57be92bb11b,client_3ffa76342f366962,425
101164,content_52c98b3f26951f3b,client_3ffa76342f366962,245
101165,content_051c8125a3d20e83,client_1a730cb2640a1abf,3
101166,content_45155b648fff917d,client_1a730cb2640a1abf,1


In [5]:
content_id = "content_8d5d1221c84f8050"
stale_df[stale_df["content"] == content_id]

,content,client,stale_days
79596,content_8d5d1221c84f8050,client_fef1a8f436438636,193


In [6]:
position_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    d.client_hash_id AS client,
    AVG(d.gsc_avg_position) AS avg_position
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN slice_df s
    USING (content_hash_id, client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
GROUP BY d.content_hash_id, d.client_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
label_df = con.execute(f"""
WITH monthly AS (
    SELECT
        d.content_hash_id AS content,
        d.client_hash_id AS client,
        AVG(CASE WHEN d.month = '2026-03'
                 THEN d.gsc_impressions END) AS avg_march,
        AVG(CASE WHEN d.month = '2026-04'
                 THEN d.gsc_impressions END) AS avg_april
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN slice_df s USING (content_hash_id, client_hash_id)
    WHERE d.month IN ('2026-03', '2026-04')
    GROUP BY d.content_hash_id, d.client_hash_id
)
SELECT
    content,
    client,
    CASE
        WHEN avg_march IS NULL OR avg_march = 0 OR avg_april IS NULL
            THEN NULL
        WHEN (avg_april - avg_march) / avg_march > 0.20
            THEN 'Up'
        WHEN (avg_april - avg_march) / avg_march < -0.20
            THEN 'Down'
        ELSE 'Stable'
    END AS label
FROM monthly
WHERE avg_march IS NOT NULL
  AND avg_march > 0
  AND avg_april IS NOT NULL
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
label_df

,content,client,label
0,content_905aa32a0230694e,client_73cda7b4e4f265ea,Down
1,content_05434271b257bb68,client_73cda7b4e4f265ea,Up
2,content_712c365258cee05c,client_73cda7b4e4f265ea,Up
3,content_476c37c366920c1b,client_73cda7b4e4f265ea,Down
4,content_3dba50ae010f3f30,client_73cda7b4e4f265ea,Down
...,...,...,...
101163,content_599d73b602d07483,client_3ffa76342f366962,Up
101164,content_908a53e9eec3b6db,client_3ffa76342f366962,Down
101165,content_8d4c42e5457c9e2e,client_3ffa76342f366962,Stable
101166,content_708b842cd69e0b38,client_3ffa76342f366962,Down


In [11]:
analysis_df = label_df.merge(stale_df, on=["content", "client"])
analysis_df = analysis_df.merge(position_df, on=["content", "client"])

analysis_df["stale_tier"] = pd.cut(
    analysis_df["stale_days"],
    bins=[0, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "180+"]
)

analysis_df["position_tier"] = pd.cut(
    analysis_df["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)

def signal_summary(signal):
    counts = analysis_df.groupby([signal, "label"], observed=False).size().unstack(fill_value=0)

    counts = counts.reindex(columns=["Down", "Up", "Stable"], fill_value=0)
    counts["Count"] = counts.sum(axis=1)

    result = counts[["Down", "Up", "Stable"]].div(counts["Count"], axis=0) * 100
    result = result.round(1)
    result["Count"] = counts["Count"]

    return result.reset_index()


stale_summary = signal_summary("stale_tier")
position_summary = signal_summary("position_tier")

display(stale_summary)
display(position_summary)


label,stale_tier,Down,Up,Stable,Count
0,0-30,33.3,48.4,18.3,8415
1,31-90,51.6,24.3,24.1,38574
2,91-180,58.9,16.7,24.4,11849
3,180+,50.1,23.3,26.5,42330


label,position_tier,Down,Up,Stable,Count
0,top_3,53.3,22.6,24.1,8986
1,page_1,49.6,26.9,23.5,46681
2,striking,51.1,24.0,24.9,21443
3,page_3_5,49.4,22.8,27.8,20287
4,deep,53.0,24.6,22.4,3771


In [12]:
ctr_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    d.client_hash_id AS client,
    SUM(d.gsc_clicks) * 1.0 / NULLIF(SUM(d.gsc_impressions), 0) AS ctr
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN slice_df s USING (content_hash_id, client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
GROUP BY d.content_hash_id, d.client_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
ctr_df

,content,client,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,0.001754
1,content_d056587ff7faca0c,client_73cda7b4e4f265ea,0.005776
2,content_2662845f598544ef,client_73cda7b4e4f265ea,0.006667
3,content_d720dde3701523c0,client_73cda7b4e4f265ea,0.007576
4,content_91ffe8aef1f8c426,client_73cda7b4e4f265ea,0.000000
...,...,...,...
101163,content_3a216d0c58d12ef3,client_3ffa76342f366962,0.002825
101164,content_48c9b57be92bb11b,client_3ffa76342f366962,0.000000
101165,content_52c98b3f26951f3b,client_3ffa76342f366962,0.090090
101166,content_051c8125a3d20e83,client_1a730cb2640a1abf,0.006593


In [14]:
analysis_ctr = label_df.merge(ctr_df, on=["content", "client"])

analysis_ctr["ctr_tier"] = pd.cut(
    analysis_ctr["ctr"],
    bins=[-0.000001, 0.01, 0.03, 0.06, float("inf")],
    labels=["Very Low", "Low", "Medium", "High"]
)

counts = analysis_ctr.groupby(
    ["ctr_tier", "label"], observed=False
).size().unstack(fill_value=0)

counts = counts.reindex(columns=["Down", "Up", "Stable"], fill_value=0)
counts["Count"] = counts.sum(axis=1)

ctr_summary = counts[["Down", "Up", "Stable"]].div(
    counts["Count"], axis=0
).mul(100).round(1)

ctr_summary["Count"] = counts["Count"]

ctr_summary = ctr_summary.reset_index()

display(ctr_summary)

label,ctr_tier,Down,Up,Stable,Count
0,Very Low,51.4,23.9,24.8,96616
1,Low,27.9,48.7,23.4,4337
2,Medium,29.0,57.0,14.0,186
3,High,51.7,37.9,10.3,29


In [15]:
analysis_peers = label_df.merge(ctr_df, on=["content", "client"])
analysis_peers = analysis_peers.merge(position_df, on=["content", "client"])

analysis_peers["position_tier"] = pd.cut(
    analysis_peers["avg_position"],
    bins=[-float("inf"), 3, 10, 20, 50, float("inf")],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)

peer_stats = analysis_peers.groupby("position_tier", observed=True)["ctr"].agg(
    peer_sum="sum",
    peer_count="count"
)

analysis_peers = analysis_peers.join(peer_stats, on="position_tier")

analysis_peers["peers_avg_ctr"] = (
    (analysis_peers["peer_sum"] - analysis_peers["ctr"]) /
    (analysis_peers["peer_count"] - 1)
)

analysis_peers["ctr_per_peers"] = (
    analysis_peers["ctr"] / analysis_peers["peers_avg_ctr"]
)

analysis_peers = analysis_peers.replace(
    [float("inf"), -float("inf")], float("nan")
).dropna(subset=["ctr_per_peers"])

In [16]:
analysis_peers["ctr_peer_tier"] = pd.cut(
    analysis_peers["ctr_per_peers"],
    bins=[-float("inf"), 0.5, 0.8, 1.2, float("inf")],
    labels=["Very Low", "Low", "Average", "High"]
)

counts = analysis_peers.groupby(
    ["ctr_peer_tier", "label"], observed=False
).size().unstack(fill_value=0)

counts = counts.reindex(columns=["Down", "Up", "Stable"], fill_value=0)
counts["Count"] = counts.sum(axis=1)

peers_summary = counts[["Down", "Up", "Stable"]].div(
    counts["Count"], axis=0
).mul(100).round(1)

peers_summary["Count"] = counts["Count"]

peers_summary = peers_summary.reset_index()

display(peers_summary)

label,ctr_peer_tier,Down,Up,Stable,Count
0,Very Low,58.0,19.8,22.3,52279
1,Low,51.4,21.4,27.1,10507
2,Average,47.2,24.5,28.3,10292
3,High,36.8,36.3,27.0,28090


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.